# 02 · Tokens and Templates: Watch a Prompt Get Built

**Hardware**: 🟢 CPU. Downloads the tokenizer and processor config only (~35MB) — **no model weights**.

## What you will do

1. Render one plain-text turn, then tokenize it
2. Add one image and watch one template marker expand into a placeholder run
3. Locate every placeholder in `input_ids` by eye
4. Compare left and right padding for batched generation
5. Optionally explore audio counts, thinking, and tools after the core path is clear

In [1]:
%pip install -q "transformers>=5.14" torch pillow numpy

In [2]:
import collections
import numpy as np
import torch
from PIL import Image
from transformers import AutoProcessor

MODEL_ID = "google/gemma-4-E2B-it"
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")
print(type(processor).__name__)

  from .autonotebook import tqdm as notebook_tqdm


Gemma4Processor


## 1. Establish the baseline: one text turn

A template is a serializer: `messages` go in, one model-specific string comes out. The tokenizer acts only after that string exists. Run this cell before introducing any multimodal placeholders.

In [ ]:
plain_messages = [{"role": "user", "content": "Explain tokenization in one sentence."}]
plain_prompt = processor.apply_chat_template(
    plain_messages, tokenize=False, add_generation_prompt=True
)
plain_ids = processor.tokenizer(plain_prompt).input_ids

print(plain_prompt.replace("<turn|>", "<turn|>\n"))
print("token count:", len(plain_ids))
print("first IDs:", plain_ids[:10])

## 2. Meet the multimodal processor

In [3]:
for name in ["tokenizer", "image_processor", "video_processor", "feature_extractor"]:
    print(f"  {name:20s} {type(getattr(processor, name)).__name__}")

print("\nprocessor defaults (audio is an optional extension below):")
print("  image_seq_length  =", processor.image_seq_length)
print("  audio_ms_per_token=", processor.audio_ms_per_token)
print("  audio_seq_length  =", processor.audio_seq_length, "(cap)")

print("\nspecial tokens:")
for attr in ["image_token", "boi_token", "eoi_token", "video_token",
             "audio_token", "boa_token", "eoa_token"]:
    tok = getattr(processor, attr, None)
    tid = processor.tokenizer.convert_tokens_to_ids(tok) if tok else None
    print(f"  {attr:12s} {str(tok):12s} -> {tid}")

  tokenizer            GemmaTokenizer
  image_processor      Gemma4ImageProcessor
  video_processor      Gemma4VideoProcessor
  feature_extractor    Gemma4AudioFeatureExtractor

processor defaults (audio is an optional extension below):
  image_seq_length  = 280
  audio_ms_per_token= 40
  audio_seq_length  = 750 (cap)

special tokens:
  image_token  <|image|>    -> 258880
  boi_token    <|image>     -> 255999
  eoi_token    <image|>     -> 258882
  video_token  <|video|>    -> 258884
  audio_token  <|audio|>    -> 258881
  boa_token    <|audio>     -> 256000
  eoa_token    <audio|>     -> 258883


Group the tokens by function before looking at their IDs. `boi_token` (`<|image>`) and `eoi_token` (`<image|>`) are learned boundary tokens. `image_token` (`<|image|>`) is the repeated placeholder whose text embedding will later be replaced. The similar spelling hides three different jobs.

Note also that `<|video|>` is **not in the shipped tokenizer**; `Gemma4Processor.__init__` adds it at construction time, with a comment in the source that says exactly that:

```python
# FIXME: add the token to config and ask Ryan to re-upload
tokenizer.add_special_tokens({"additional_special_tokens": ["<|video|>"]})
```

## 3. Add one image: the template emits one marker

This is the most common misunderstanding about multimodal chat templates.

In [4]:
messages = [{
    "role": "user",
    "content": [
        {"type": "image"},
        {"type": "text", "text": "What is in this image?"},
    ],
}]

rendered = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(repr(rendered))
print("\n<|image|> count in the rendered string:", rendered.count(processor.image_token))

'<bos><|turn>user\n<|image|>What is in this image?<turn|>\n<|turn>model\n'

<|image|> count in the rendered string: 1


**One.** The expansion happens later, in the processor, once the image processor has actually looked at the image and decided how many soft tokens it is worth:

```python
def replace_image_token(self, image_inputs, image_idx):
    num_soft_tokens = image_inputs["num_soft_tokens_per_image"][image_idx]
    return f"{self.boi_token}{self.image_token * num_soft_tokens}{self.eoi_token}"
```

That ordering is why the processor cannot produce a **model-ready** multimodal batch without the attachment. You can still tokenize the one-marker string as text, but it does not yet reserve the `N` positions required by the image features.

In [5]:
image = Image.fromarray(np.random.randint(0, 256, (240, 320, 3), dtype=np.uint8))

batch = processor(text=[rendered], images=[[image]], return_tensors="pt")
for k, v in batch.items():
    print(f"  {k:22s} {tuple(v.shape)}")

ids = batch["input_ids"][0]
n_img = int((ids == processor.image_token_id).sum())
print(f"\nrendered string had 1 marker; input_ids has {n_img} image placeholders")
print(f"total sequence length: {len(ids)}  ({len(ids) - n_img} text tokens)")
print("\n266, not 280 — a 320x240 image does not fill the default budget (chapter 03).")

  input_ids              (1, 283)
  attention_mask         (1, 283)
  mm_token_type_ids      (1, 283)
  pixel_values           (1, 2520, 768)
  image_position_ids     (1, 2520, 2)

rendered string had 1 marker; input_ids has 266 image placeholders
total sequence length: 283  (17 text tokens)

266, not 280 — a 320x240 image does not fill the default budget (chapter 03).


## 4. Find the placeholders by eye

In [6]:
toks = processor.tokenizer.convert_ids_to_tokens(ids.tolist())

def runs(seq):
    out = []
    for t in seq:
        if out and out[-1][0] == t:
            out[-1][1] += 1
        else:
            out.append([t, 1])
    return out

print(f"{'token':<16} {'id':>8}  count")
print("-" * 36)
for tok, n in runs(toks):
    tid = processor.tokenizer.convert_tokens_to_ids(tok)
    marker = "  <-- placeholder run" if n > 3 else ""
    print(f"{tok!r:<16} {tid:>8}  x{n}{marker}")

token                  id  count
------------------------------------
'<bos>'                 2  x1
'<|turn>'             105  x1
'user'               2364  x1
'\n'                  107  x1
'<|image>'         255999  x1
'<|image|>'        258880  x266  <-- placeholder run
'<image|>'         258882  x1
'What'               3689  x1
'▁is'                 563  x1
'▁in'                 528  x1
'▁this'               672  x1
'▁image'             2471  x1
'?'                236881  x1
'<turn|>'             106  x1
'\n'                  107  x1
'<|turn>'             105  x1
'model'              4368  x1
'\n'                  107  x1


In [7]:
# mm_token_type_ids labels each position by modality -- chapter 08 turns this into block ids
counts = collections.Counter(batch["mm_token_type_ids"][0].tolist())
names = {0: "text", 1: "image", 2: "video", 3: "audio"}
print("mm_token_type_ids:", {names.get(k, k): v for k, v in sorted(counts.items())})
print("\nGemma4Processor.model_input_names appends 'mm_token_type_ids' precisely so")
print("the model can rebuild block structure without re-scanning input_ids.")

mm_token_type_ids: {'text': 17, 'image': 266}

Gemma4Processor.model_input_names appends 'mm_token_type_ids' precisely so
the model can rebuild block structure without re-scanning input_ids.


## Optional extension A: audio counts

The core image path is complete. This optional cell gives the audio rule a concrete shape without requiring you to understand the audio tower: call the processor's counting helper for several waveform lengths and compare the result with the 40 ms approximation.

In [8]:
print(f"{'duration':>9} | {'tokens':>7} | {'duration_ms / 40':>17}")
print("-" * 40)
for sec in (0.5, 1.0, 3.4, 10.0, 30.0, 40.0, 60.0):
    n = processor._compute_audio_num_tokens(np.zeros(int(sec * 16000)), 16000)
    print(f"{sec:8.1f}s | {n:7d} | {sec * 1000 / 40:17.0f}")

print(f"\naudio_seq_length cap = {processor.audio_seq_length} tokens = "
      f"{processor.audio_seq_length * 40 / 1000:.0f}s of audio")
print("Past 30s the count saturates. Longer clips must be chunked by you.")

 duration |  tokens |  duration_ms / 40
----------------------------------------
     0.5s |      13 |                12
     1.0s |      25 |                25
     3.4s |      85 |                85
    10.0s |     250 |               250
    30.0s |     750 |               750
    40.0s |     750 |              1000
    60.0s |     750 |              1500

audio_seq_length cap = 750 tokens = 30s of audio
Past 30s the count saturates. Longer clips must be chunked by you.


The small short-duration deviation comes from mel framing. The only contract needed here is that the placeholder count must match the audio tower's valid output length. Chapter 05 derives the exact formula from the encoder; return to `_compute_audio_num_tokens` after reading it.

## Optional extension B: system turns and thinking

In [9]:
def show(label, **kw):
    msgs = kw.pop("messages")
    out = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **kw)
    print(f"--- {label} ---")
    print(out.replace("<turn|>", "<turn|>\n").rstrip())
    print()

plain = [{"role": "user", "content": [{"type": "text", "text": "hi"}]}]
show("plain", messages=plain)
show("with a system message",
     messages=[{"role": "system", "content": "You are terse."}] + plain)
show("enable_thinking=True", messages=plain, enable_thinking=True)

--- plain ---
<bos><|turn>user
hi<turn|>

<|turn>model

--- with a system message ---
<bos><|turn>system
You are terse.<turn|>

<|turn>user
hi<turn|>

<|turn>model

--- enable_thinking=True ---
<bos><|turn>system
<|think|>
<turn|>

<|turn>user
hi<turn|>

<|turn>model



Two things to notice.

`enable_thinking=True` is **not a generation flag** — it injects a `<|think|>` token at the top of the first system turn. And the system turn is *synthesised* if it does not exist: the template emits one if there is a system message, **or** tools, **or** thinking is enabled.

## Optional extension C: tool declarations

In [10]:
WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_n_day_weather_forecast",
        "description": "Get an N-day weather forecast",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The city and state"},
                "format": {"type": "string", "enum": ["celsius", "fahrenheit"],
                           "description": "The temperature unit to use"},
                "num_days": {"type": "integer", "description": "The number of days"},
            },
            "required": ["location", "format", "num_days"],
        },
    },
}

with_tools = processor.apply_chat_template(
    [{"role": "user", "content": "Weather in SF for 3 days?"}],
    tools=[WEATHER_TOOL], tokenize=False, add_generation_prompt=True)
print(with_tools.replace("<tool|>", "<tool|>\n").replace("<turn|>", "<turn|>\n"))

<bos><|turn>system
<|tool>declaration:get_n_day_weather_forecast{description:<|"|>Get an N-day weather forecast<|"|>,parameters:{properties:{format:{description:<|"|>The temperature unit to use<|"|>,enum:[<|"|>celsius<|"|>,<|"|>fahrenheit<|"|>],type:<|"|>STRING<|"|>},location:{description:<|"|>The city and state<|"|>,type:<|"|>STRING<|"|>},num_days:{description:<|"|>The number of days<|"|>,type:<|"|>INTEGER<|"|>}},required:[<|"|>location<|"|>,<|"|>format<|"|>,<|"|>num_days<|"|>],type:<|"|>OBJECT<|"|>}}<tool|>
<turn|>

<|turn>user
Weather in SF for 3 days?<turn|>

<|turn>model



In [11]:
# Structural punctuation is made of single tokens, not literal characters.
import json
as_json = json.dumps(WEATHER_TOOL)
n_json = len(processor.tokenizer(as_json).input_ids)
n_dsl = len(processor.tokenizer(with_tools).input_ids)
n_bare = len(processor.tokenizer(processor.apply_chat_template(
    [{"role": "user", "content": "Weather in SF for 3 days?"}],
    tokenize=False, add_generation_prompt=True)).input_ids)

print(f"tool declaration as raw JSON    : {n_json} tokens")
print(f"tool declaration in Gemma's DSL : {n_dsl - n_bare} tokens")
print(f"saving                          : {100 * (1 - (n_dsl - n_bare) / n_json):.0f}%")
print()
print("This size comparison is optional implementation detail; callers should pass")
print("normal JSON-schema dictionaries and let the chat template serialize them.")

tool declaration as raw JSON    : 131 tokens
tool declaration in Gemma's DSL : 123 tokens
saving                          : 6%

This size comparison is optional implementation detail; callers should pass
normal JSON-schema dictionaries and let the chat template serialize them.


## Practical epilogue: left padding for batched generation

A decoder-only generation loop reads next-token logits from the final tensor column. Left padding makes that column a real prompt token for every sample. This is a batched-generation convention, not a universal rule: right padding remains common in training and encoder-only workloads.

In [12]:
prompts = [
    processor.apply_chat_template([{"role": "user", "content": "hi"}],
                                  tokenize=False, add_generation_prompt=True),
    processor.apply_chat_template([{"role": "user", "content": "tell me about tokenizers in detail"}],
                                  tokenize=False, add_generation_prompt=True),
]

for side in ("left", "right"):
    tok = AutoProcessor.from_pretrained(MODEL_ID, padding_side=side).tokenizer
    enc = tok(prompts, return_tensors="pt", padding=True)
    print(f"padding_side={side}:")
    for row, mask in zip(enc["input_ids"], enc["attention_mask"]):
        last_real = int(mask.nonzero()[-1])
        print(f"   len={len(row)}  last real token at index {last_real}")
    print()

print("Left padding puts every sequence's last real token at the same index.")
print("For mixed-length decoder-only generation, this is the layout generate() expects.")

padding_side=left:
   len=16  last real token at index 15
   len=16  last real token at index 15



padding_side=right:
   len=16  last real token at index 9
   len=16  last real token at index 15

Left padding puts every sequence's last real token at the same index.
For mixed-length decoder-only generation, this is the layout generate() expects.


## After generation: slice the prompt off

`generate` returns prompt + completion, always.

In [13]:
input_len = batch["input_ids"].shape[-1]
print(f"input_len = {input_len}")
for line in [
    "",
    "    output = model.generate(**inputs, max_new_tokens=50)",
    "    text   = processor.decode(output[0][input_len:], skip_special_tokens=True)",
    "                                        ^^^^^^^^^^^",
    "",
    "skip_special_tokens is a decision, not a default:",
    "  True  -> clean prose",
    "  False -> you can see <|turn>, <|channel>thought, <|tool_call>,",
    "           which is the only way to debug tool calling or read thinking output",
]:
    print(line)

input_len = 283

    output = model.generate(**inputs, max_new_tokens=50)
    text   = processor.decode(output[0][input_len:], skip_special_tokens=True)
                                        ^^^^^^^^^^^

skip_special_tokens is a decision, not a default:
  True  -> clean prose
  False -> you can see <|turn>, <|channel>thought, <|tool_call>,
           which is the only way to debug tool calling or read thinking output


## Exercises

1. Two images in one message. How many placeholder runs appear, and what separates them?
2. Pass `images=[img1, img2]` (flat) instead of `[[img1, img2]]` (nested) and read the error from `validate_inputs`. Why does it check per sample?
3. Render an assistant turn containing `reasoning`, then render it again after a later user message. Where did the reasoning go? Find `strip_thinking` and `thinking_gate` in the template.
4. Pass `tool_calls` with `arguments` as a JSON **string**. The template raises deliberately — find the `raise_exception` and explain the design choice.
5. Compute how many audio tokens a 90-second podcast clip would need, and what you must do about it.
6. Resize the image to 1024×1024 and re-run section 2. How many placeholders now, and why did the *text* length change?

## Where next

[Chapter 03](../../03-image-processor/index.md) explains where 266 came from.